# Gold labeling — manual review of silver labels

Assign a human `gold_label` (1 = same patient, 0 = different) to each silver 
pair, as fast as possible. Each pair renders as a **stacked, color-coded** 
table — record **A above**, record **B below**, one column per field:

<span style='background:#b7e1a1'>&nbsp;green&nbsp;</span> both equal &nbsp;·&nbsp; 
<span style='background:#f4a6a6'>&nbsp;red&nbsp;</span> different &nbsp;·&nbsp; 
<span style='background:#f7e59b'>&nbsp;yellow&nbsp;</span> one missing &nbsp;·&nbsp; 
<span style='background:#d9d9d9'>&nbsp;grey&nbsp;</span> both missing.

Workflow: (1) load silver + records, (2) **bulk-accept** the confident buckets 
from the EDA triage CSV, (3) hand-review the rest with the keyboard/button 
labeler. Labels autosave to `GOLD_CSV` and reload on restart, so it's resumable.

> PHI: run on HIPAA-tier compute only. Edit paths in **Config**.

In [ ]:
# --- Bootstrap: run from anywhere, hop to the AnyMatch repo root -------------
import os, sys

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # Windows OpenMP guard (see CLAUDE.md)

def _find_repo_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.exists(os.path.join(d, 'loo.py')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError("Could not find repo root (no loo.py above this notebook).")
        d = parent

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
from IPython.display import HTML, display

from utils import labeling_utils as L

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)
print('repo root:', REPO_ROOT)

## Config

In [ ]:
SILVER_CSV      = 'data/silver_labels/silver_labels_v1_2026_06_21.csv'
RECORDS_PARQUET = 'data/alliance/MDM_Population_cleaned_v3_2026_06_11.parquet'
TRIAGE_CSV      = 'silver_labels/silver_labels_triage_v1.csv'  # from silver_labels_eda (optional)
GOLD_CSV        = 'gold_labeling/gold_labels_v1.csv'           # output (resumable)

## 1. Load silver labels + records, merge to per-field columns

In [ ]:
# Read silver labels. get_silver_labels.ipynb wrote with the index, so drop any
# unnamed index column and force the PATID columns to string (ID dtype trap).
silver = pd.read_csv(SILVER_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'})
silver = silver.loc[:, ~silver.columns.str.startswith('Unnamed')]
silver['silver_label'] = (silver['silver_label'].astype('string').str.strip().str.lower()
                          .map({'true': True, 'false': False, '1': True, '0': False}))
silver = silver[['PATID_A', 'PATID_B', 'silver_label']].drop_duplicates(['PATID_A', 'PATID_B'])
print(f'{len(silver):,} silver pairs')
print(silver['silver_label'].value_counts(dropna=False))
silver.head()

In [ ]:
# Load valid records (friendly schema) and attach <field>_a / <field>_b to every pair.
records = L.load_records(RECORDS_PARQUET)
print(f'{len(records):,} valid records; PATID index unique = {records.index.is_unique}')

pairs = L.merge_pairs(silver, records)
joinable = L.joinable_mask(pairs)
print(f'{len(pairs):,} pairs; {(~joinable).sum():,} not joinable to a valid record on one/both sides')

## 2. Initialize / restore the `gold_label` column

`gold_label` is nullable Int (1 / 0 / <NA>). If `GOLD_CSV` already exists its 
labels are merged back in so you continue where you left off. If the triage CSV 
exists, its `bucket` / `suggested_gold` ride along to drive bulk-accept and 
review ordering.

In [ ]:
pairs['gold_label'] = pd.Series(pd.NA, index=pairs.index, dtype='Int64')
key = ['PATID_A', 'PATID_B']
pairs = pairs.set_index(key, drop=False)

# Attach triage info if present.
if os.path.exists(TRIAGE_CSV):
    tri = pd.read_csv(TRIAGE_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'}).set_index(key)
    pairs['bucket'] = tri['bucket'].reindex(pairs.index)
    pairs['suggested_gold'] = tri['suggested_gold'].reindex(pairs.index).astype('Int64')
    pairs['priority'] = tri['priority'].reindex(pairs.index).fillna(9).astype(int)
    print('triage attached:', pairs['bucket'].value_counts().to_dict())
else:
    pairs['bucket'] = 'review'
    pairs['suggested_gold'] = pd.Series(pd.NA, index=pairs.index, dtype='Int64')
    pairs['priority'] = 0
    print('no triage CSV; every pair goes to manual review')

# Restore any labels already saved.
if os.path.exists(GOLD_CSV):
    prev = pd.read_csv(GOLD_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'}).set_index(key)
    pairs['gold_label'] = prev['gold_label'].reindex(pairs.index).astype('Int64')
    print(f'restored {pairs["gold_label"].notna().sum():,} existing gold labels from {GOLD_CSV}')

def save_gold():
    out = pairs.reset_index(drop=True)[['PATID_A', 'PATID_B', 'silver_label', 'gold_label']]
    os.makedirs(os.path.dirname(GOLD_CSV), exist_ok=True)
    out.to_csv(GOLD_CSV, index=False)

def progress():
    done = pairs['gold_label'].notna().sum()
    print(f'labeled {done:,} / {len(pairs):,} ({done/len(pairs):.1%})  |  remaining {len(pairs)-done:,}')

save_gold(); progress()

## 3. The visualizer

`show(pair_id)` or `show(i)` renders one pair; `show_many(...)` stacks several.

In [ ]:
display(HTML(L.legend_html()))

def _row_for(ref):
    """Accept a positional int, a (PATID_A, PATID_B) tuple, or a MultiIndex key."""
    if isinstance(ref, tuple):
        return pairs.loc[ref]
    if isinstance(ref, (int, np.integer)):
        return pairs.iloc[int(ref)]
    return pairs.loc[ref]

def show(ref):
    row = _row_for(ref)
    display(HTML(L.render_pair_html(row)))

def show_many(refs):
    rows = [_row_for(r) for r in refs]
    display(HTML(''.join(L.render_pair_html(r) for r in rows)))

# Example: first 3 pairs flagged suspicious (a missed match, if any).
susp = pairs[pairs['bucket'] == 'suspicious_neg']
show_many(range(min(3, len(susp))) and susp.index[:3]) if len(susp) else show(0)

## 4. Bulk-accept the confident buckets

The EDA already separated pairs the field evidence makes obvious. **Spot-check** 
a sample of each confident bucket below, then accept the whole bucket with one 
call. Re-run with `confirm=True` to actually write.

In [ ]:
# Spot-check before trusting a bucket wholesale.
for b in ['confident_match', 'confident_nonmatch']:
    sub = pairs[pairs['bucket'] == b]
    print(f'--- {b}: {len(sub):,} pairs, sample of 4 ---')
    if len(sub):
        display(HTML(''.join(L.render_pair_html(r) for _, r in sub.sample(min(4, len(sub)), random_state=0).iterrows())))

In [ ]:
def bulk_accept(bucket_name, label, confirm=False, only_unlabeled=True):
    """Set gold_label=label for every pair in a triage bucket."""
    mask = pairs['bucket'] == bucket_name
    if only_unlabeled:
        mask &= pairs['gold_label'].isna()
    n = int(mask.sum())
    if not confirm:
        print(f'[dry run] would set {n:,} `{bucket_name}` pairs -> gold_label={label}. Pass confirm=True.')
        return
    pairs.loc[mask, 'gold_label'] = label
    save_gold()
    print(f'set {n:,} `{bucket_name}` pairs -> {label}')
    progress()

# Dry runs (flip confirm=True when you trust the spot-check):
# bulk_accept('confident_match', 1, confirm=False)
# bulk_accept('confident_nonmatch', 0, confirm=False)

## 5. Fast manual labeler

Steps through the **review queue** (everything not bulk-accepted, most-suspicious 
first). Buttons: **Match (1)**, **Non-match (0)**, **Unsure** (skip, leave <NA>), 
**Back**. Every click autosaves and advances. Needs `ipywidgets`; if it's 
unavailable, use the keyboardless fallback in the next cell.

In [ ]:
# Build the review queue: unlabeled pairs, suspicious/review first.
def build_queue():
    q = pairs[pairs['gold_label'].isna()].copy()
    q = q.sort_values(['priority', 'PATID_A', 'PATID_B'])
    return list(q.index)

queue = build_queue()
print(f'{len(queue):,} pairs in the review queue')

In [ ]:
try:
    import ipywidgets as widgets
    _HAVE_WIDGETS = True
except Exception as e:
    _HAVE_WIDGETS = False
    print('ipywidgets unavailable -> use the fallback cell. (', e, ')')

if _HAVE_WIDGETS:
    state = {'pos': 0}
    out = widgets.Output()
    status_lbl = widgets.HTML()

    def _render():
        out.clear_output(wait=True)
        with out:
            if not queue:
                print('queue empty - nothing to review'); return
            state['pos'] = max(0, min(state['pos'], len(queue) - 1))
            key = queue[state['pos']]
            row = pairs.loc[key]
            cur = row['gold_label']
            cur_txt = '<NA>' if pd.isna(cur) else int(cur)
            display(HTML(L.legend_html()))
            display(HTML(L.render_pair_html(row)))
            print(f'queue {state["pos"]+1}/{len(queue)}  bucket={row["bucket"]}  '
                  f'silver={row["silver_label"]}  current gold={cur_txt}')
        done = pairs['gold_label'].notna().sum()
        status_lbl.value = f'<b>labeled {done:,}/{len(pairs):,}</b> ({done/len(pairs):.1%})'

    def _set(label):
        if not queue: return
        key = queue[state['pos']]
        pairs.loc[key, 'gold_label'] = label
        save_gold()
        state['pos'] += 1
        _render()

    b_match = widgets.Button(description='Match (1)', button_style='success')
    b_non   = widgets.Button(description='Non-match (0)', button_style='danger')
    b_uns   = widgets.Button(description='Unsure / skip', button_style='warning')
    b_back  = widgets.Button(description='← Back')
    b_match.on_click(lambda _: _set(1))
    b_non.on_click(lambda _: _set(0))
    def _skip(_):
        state['pos'] += 1; _render()
    def _back(_):
        state['pos'] -= 1; _render()
    b_uns.on_click(_skip)
    b_back.on_click(_back)

    display(widgets.HBox([b_match, b_non, b_uns, b_back, status_lbl]))
    display(out)
    _render()

### Fallback labeler (no ipywidgets)

Label by hand: render a window, then assign with `label_pair(...)`.

In [ ]:
def label_pair(ref, value, save=True):
    """Set one pair's gold_label by positional int or (PATID_A,PATID_B) key."""
    row = _row_for(ref)
    pairs.loc[(row['PATID_A'], row['PATID_B']), 'gold_label'] = value
    if save: save_gold()

def review_window(start=0, n=5):
    """Render the next n unlabeled queue pairs with their queue positions."""
    sl = queue[start:start + n]
    for i, key in enumerate(sl, start=start):
        print(f'--- queue #{i}  key={key} ---')
        display(HTML(L.render_pair_html(pairs.loc[key])))
    return sl

# Example:
# win = review_window(0, 5)
# label_pair(win[0], 1)   # mark the first as a match

## 6. Progress & export

In [ ]:
progress()
print()
print('gold_label distribution:')
print(pairs['gold_label'].value_counts(dropna=False))
print()
if 'bucket' in pairs:
    print('labeled vs bucket:')
    print(pd.crosstab(pairs['bucket'], pairs['gold_label'].fillna('unlabeled')))
save_gold()
print(f'\nsaved -> {GOLD_CSV}')